In [135]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,BatchNormalization,LSTM,Input
from tensorflow.keras.losses import MeanAbsoluteError,MeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import MinMaxScaler
import yfinance
from datetime import datetime
from sklearn.model_selection import train_test_split
import pandas as pd

In [6]:
ticker = yfinance.Ticker("NVDA")

In [68]:
start_date = datetime(2020, 1, 1)
end_date = datetime(2026, 1, 1)
data = ticker.history(start="2020-01-01",end="2026-01-01")

In [69]:
delta = data['Close'].diff()
gain = delta.clip(lower=0)
loss = -1 * delta.clip(upper=0)

data['Avg_gain'] = gain.rolling(14,min_periods=1).mean()
data['Avg_loss'] = loss.rolling(14,min_periods=1).mean()
rs = data['Avg_gain'] / data['Avg_loss']
data['RSI'] = 100 - (100 / (1 + rs))
data = data[1:]

exp1 = data['Close'].ewm(span=12, adjust=False).mean()
exp2 = data['Close'].ewm(span=26, adjust=False).mean()
data['MACD'] = exp1 - exp2
data['Signal_Line'] = data['MACD'].ewm(span=9, adjust=False).mean()

data['MA20'] = data['Close'].rolling(window=20,min_periods=1).mean()
data['Upper_Band'] = data['MA20'] + (data['Close'].rolling(window=20,min_periods=1).std() * 2)
data['Lower_Band'] = data['MA20'] - (data['Close'].rolling(window=20,min_periods=1).std() * 2)
data = data[1:]
data.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1506 entries, 2020-01-06 00:00:00-05:00 to 2025-12-31 00:00:00-05:00
Data columns (total 15 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Open          1506 non-null   float64
 1   High          1506 non-null   float64
 2   Low           1506 non-null   float64
 3   Close         1506 non-null   float64
 4   Volume        1506 non-null   int64  
 5   Dividends     1506 non-null   float64
 6   Stock Splits  1506 non-null   float64
 7   Avg_gain      1506 non-null   float64
 8   Avg_loss      1506 non-null   float64
 9   RSI           1506 non-null   float64
 10  MACD          1506 non-null   float64
 11  Signal_Line   1506 non-null   float64
 12  MA20          1506 non-null   float64
 13  Upper_Band    1506 non-null   float64
 14  Lower_Band    1506 non-null   float64
dtypes: float64(14), int64(1)
memory usage: 188.2 KB


In [70]:
data = data.drop(columns=["Dividends","Stock Splits"])

In [120]:
scaler_X= MinMaxScaler()
scaler_Y= MinMaxScaler()
normalized_y = scaler_Y.fit_transform(data["Close"].values.reshape(-1,1))
data = data.drop(columns=['Close'])
normalized_x = scaler_X.fit_transform(data)
X,y =[],[]

for i in range(len(normalized_x)-7):
  X.append(normalized_x[i:i+7])
  y.append(normalized_y[i+7])
X = np.array(X)
y = np.array(y)

X_train , X_test, y_train, y_test = train_test_split(X,y,shuffle=False)

In [125]:
X_train.shape

(1124, 7, 12)

In [129]:
Model = Sequential(
    [
        Input((7,12)),
        LSTM(150,activation='relu'),
        Dropout(0.2),
        Dense(1,activation="linear")
    ]
)

In [130]:
Model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss=MeanSquaredError(),
    metrics=[MeanAbsoluteError()]
)

In [166]:
Model.fit(X_train,y_train,epochs=50)

Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.2955e-04 - mean_absolute_error: 0.0095
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2.3803e-04 - mean_absolute_error: 0.0099
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2.1172e-04 - mean_absolute_error: 0.0090
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.8947e-04 - mean_absolute_error: 0.0092
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 2.6777e-04 - mean_absolute_error: 0.0105
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.6281e-04 - mean_absolute_error: 0.0094
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 2.2291e-04 - mean_absolute_error: 0.0094
Epoch 8/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 2.2579e-04 - mean_absolute_error: 0.0093
Epoch 9/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - loss: 1.8324e-04 - mean_absolute_error: 0.0089
Epoch 10/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 1.9390e-04 - mean_absolute_error: 0.0087

In [167]:
preds = Model.predict(X_test)


12/12 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step


In [168]:
preds_scaled = scaler_Y.inverse_transform(preds)
y_test_ = scaler_Y.inverse_transform(y_test)


In [169]:
res = pd.DataFrame({
    "Prediction" : preds_scaled.flatten(),
    "True Values" : y_test_.flatten()
})

In [170]:
res['Difference'] = abs(res['True Values'] - res['Prediction'])
avg_loss = res['Difference'].mean()
print(f"AVerage loss : {avg_loss}")
print(res.head())

AVerage loss : 5.278053283691406
   Prediction  True Values  Difference
0  123.600464   125.776169    2.175705
1  123.162224   128.145142    4.982918
2  123.249809   131.323807    8.073997
3  124.741463   134.852295   10.110832
4  126.052826   127.345512    1.292686
